In [1]:
import pandas as pd
import joblib
import pulp

In [2]:
df = pd.read_csv("../data/features/feature_dataset.csv")

model = joblib.load("../models/tuned_xgboost_model.pkl")

In [3]:
X = df.drop("Disruption_Occurred", axis=1)

In [4]:
X = X.drop(columns=["High_Risk"], errors="ignore")

In [5]:
df["Prediction"] = model.predict(X)

df["Probability"] = model.predict_proba(X)[:, 1]

In [6]:
high_risk = df[df["Probability"] > 0.80]

In [7]:
problem = pulp.LpProblem(
    "Shipment_Optimization",
    pulp.LpMinimize
)

In [8]:
decision = pulp.LpVariable.dicts(
    "Shipment",
    high_risk.index,
    cat="Binary"
)

In [9]:
problem += pulp.lpSum(
    decision[i]
    for i in high_risk.index
)

In [10]:
for i in high_risk.index:
    problem += decision[i] == 1

In [11]:
problem.solve()

1

In [12]:
high_risk["Recommended_Action"] = high_risk.index.map(
    lambda i: (
        "Prioritize Shipment"
        if decision[i].value() == 1
        else "Normal Processing"
    )
)

In [13]:
high_risk[
    [
        "Prediction",
        "Probability",
        "Recommended_Action"
    ]
].head()

,Prediction,Probability,Recommended_Action
0,1,0.963058,Prioritize Shipment
1,1,0.905390,Prioritize Shipment
3,1,0.945838,Prioritize Shipment
14,1,0.866301,Prioritize Shipment
15,1,0.935500,Prioritize Shipment


In [14]:
import os

os.makedirs("../recommendations", exist_ok=True)

high_risk.to_csv(
    "../recommendations/recommendations.csv",
    index=False
)